# 20 — Pilota del few-shot: k e variante di prompt

Sceglie la configurazione del notebook 21 (`qwen_fewshot`) prima di spendere ~40 h di
generazione sul test. Rifà, con un'impostazione corretta, l'esplorazione di Federica in
`notebooks/llm/20_preliminary_fs_test.ipynb` (archiviata con i suoi output), di cui tiene i
risultati come punto di partenza:

- **embedding**: solo `all-mpnet-base-v2` (`sbert_mpnet_en`), il migliore dei quattro nel suo
  pilota (TF-IDF, SBERT multilingue, SBERT inglese, `nomic-embed-text`); non si rifà il
  confronto, le differenze fra embedding erano minime;
- **griglia**: k ∈ {2, 4} × prompt {`old`, `new`} (la variante `structured` era la peggiore su
  ROUGE e METEOR ed è scartata), più il **controllo k=0**: il prompt zero-shot del notebook 07,
  mandato con la stessa pipeline (API nativa, `num_ctx`). Il confronto con k=0 isola l'effetto
  degli esempi da quello di tutto il resto;
- **baseline retrieval-only**: il riassunto del 1° vicino usato *come* riassunto. Misura quanto
  il retrieval da solo si avvicina al riferimento — se molto, il few-shot rischia di copiare la
  notizia gemella del train invece di riassumere il documento.

Tre cose che il pilota originale non controllava e questo sì:
1. **Righe di validation, non test né campione.** Il campione condiviso contiene righe del
   train, che ritroverebbero se stesse come esempio; pilotare sul test vorrebbe dire scegliere
   la configurazione guardando i dati su cui la si valuta. Qui: 100 righe della split `val`
   (seed 42); il pool degli esempi è il train, come nel notebook 21.
2. **Contesto di ollama.** I prompt con k=4 arrivano a decine di migliaia di token; il pilota
   originale usava l'endpoint di default, che su versioni/macchine con contesto piccolo li
   tronca in silenzio. Qui `num_ctx=32768` e una richiesta che satura il contesto fallisce
   (`su.genera_ollama_nativo`).
3. **Rumore.** Con 100 righe l'errore standard di ROUGE-1 F1 è ~0,009: la scelta si fa sulle
   differenze appaiate rispetto a k=0 con il loro IC 95%, non su un punteggio composito
   normalizzato min-max (che amplifica differenze dentro il rumore).

In [ ]:
# --- Configurazione ---------------------------------------------------------
import json
import os
import random

import summ_utils as su

# PILOTA_N=3 per uno smoke test: gli output vanno in una cartella propria (pilota_val3/)
N_PILOTA   = int(os.environ.get('PILOTA_N', 100))
SEED       = 42
MODELLO    = 'qwen2.5:7b-instruct'
MAX_TOKENS = 200          # come i notebook 07 e 21
TEMPERATURE = 0.3
K_MAX      = 4
CONFIGURAZIONI = {'k0': (0, None),
                  'k2_old': (2, 'old'), 'k2_new': (2, 'new'),
                  'k4_old': (4, 'old'), 'k4_new': (4, 'new')}

BASE = su.trova_base_dir()
P    = su.percorsi_standard(BASE)
PILOTA_DIR  = BASE / 'results' / 'fewshot' / f'pilota_val{N_PILOTA}'
VICINI_PATH = PILOTA_DIR / 'vicini.tsv'
CACHE_EMB   = BASE / 'results' / 'embeddings_cache'
RISULTATI_PATH = PILOTA_DIR / 'riepilogo.json'

# Controllo k=0: il prompt zero-shot del notebook 07, letto dalla config della sua corsa
# sul test cosi' e' per costruzione lo stesso testo
_cfg07 = json.load(open(P['metrics_dir'] / 'qwen_test_aggregate.json', encoding='utf-8'))['config']
PROMPT_SYSTEM_07, PROMPT_USER_07 = _cfg07['prompt_system'], _cfg07['prompt_user']

righe_val = list(su.itera_split(P['complete_tab'], 'val'))
righe = sorted(random.Random(SEED).sample(righe_val, N_PILOTA), key=lambda r: r['row_id'])
del righe_val
print(f'{len(righe)} righe di validation, row_id {righe[0]["row_id"]}..{righe[-1]["row_id"]}')

## Retrieval

In [ ]:
if VICINI_PATH.exists():
    VICINI = su.carica_vicini(VICINI_PATH)
else:
    VICINI = su.calcola_vicini(P['complete_tab'], righe, K_MAX, CACHE_EMB,
                               f'pilota_val{N_PILOTA}', VICINI_PATH)
ESEMPI_TRAIN = su.carica_esempi_train(P['complete_tab'],
                                      {v for r in righe for v, _ in VICINI[r['row_id']]})
# Riassunto del 1o vicino, intero (per la baseline retrieval-only)
PRIMO_VICINO = {r['row_id']: ESEMPI_TRAIN[VICINI[r['row_id']][0][0]]['summary'] for r in righe}

## Generazione

Un TSV per configurazione in `results/fewshot/pilota_val100/`, con la stessa ripresa dei notebook
dei metodi: rieseguire la cella completa solo le righe mancanti.

In [ ]:
TOKEN_PROMPT = {c: [] for c in CONFIGURAZIONI}

def messaggi(es, k, variante):
    documento = su.tronca_parole(su.prepara_documento(es['document']),
                                 su.MAX_PAROLE_DOCUMENTO_FEWSHOT)
    if k == 0:
        return [{'role': 'system', 'content': PROMPT_SYSTEM_07},
                {'role': 'user', 'content': PROMPT_USER_07.format(documento=documento)}]
    esempi = [ESEMPI_TRAIN[v] for v, _ in VICINI[es['row_id']][:k]]
    return [{'role': 'user', 'content': su.prompt_fewshot(esempi, documento, variante)}]

for nome, (k, variante) in CONFIGURAZIONI.items():
    # Il "documento" passato al ciclo e' la lista di messaggi gia' pronta
    def righe_config():
        for es in righe:
            yield {**es, 'document': messaggi(es, k, variante)}

    def genera(msg, nome=nome):
        testo, n = su.genera_ollama_nativo(msg, MODELLO, MAX_TOKENS, TEMPERATURE)
        TOKEN_PROMPT[nome].append(n)
        return testo

    scrittore = su.ScrittoreRiassunti(PILOTA_DIR / f'{nome}.tsv')
    su.ciclo_summarization(righe_config(), scrittore, genera, etichetta=f'{nome} ',
                           prepara=lambda m: m)
    scrittore.chiudi()

for nome, t in TOKEN_PROMPT.items():
    if t:
        t = sorted(t)
        print(f'{nome:7s} token del prompt: mediana {t[len(t) // 2]}, max {t[-1]}')

## Metriche

Stesse metriche del benchmark (`su.metriche_esempio`) più BERTScore (`roberta-large`). Per
ogni configurazione: media ± errore standard, e differenza **appaiata** rispetto a k=0 (stesse
righe) con IC 95%.

In [ ]:
import numpy as np
import pandas as pd

valutatore = su.crea_valutatore()
riferimenti = {r['row_id']: su.pulisci_riferimento(r['summary']) for r in righe}
predizioni = {nome: su.carica_riassunti(PILOTA_DIR / f'{nome}.tsv') for nome in CONFIGURAZIONI}
predizioni['retrieval_only'] = PRIMO_VICINO

coppie, righe_metriche = [], []
for nome, pred in predizioni.items():
    for rid, testo in pred.items():
        righe_metriche.append({'config': nome, 'row_id': rid,
                               **su.metriche_esempio(valutatore, testo, riferimenti[rid])})
        coppie.append((f'{nome}|{rid}', testo, riferimenti[rid]))
bs = su.calcola_bertscore_batch(coppie)
df = pd.DataFrame(righe_metriche)
df['bertscore_f1'] = [bs[f'{c}|{r}']['bertscore_f1'] for c, r in zip(df.config, df.row_id)]

METRICHE = ['rouge1_f1', 'rouge2_f1', 'rougeL_f1', 'meteor', 'bertscore_f1', 'parole_generate']
ordine = list(predizioni)
media = df.groupby('config')[METRICHE].mean().loc[ordine]
se = (df.groupby('config')[METRICHE].std() / np.sqrt(df.groupby('config').size().values[:, None])).loc[ordine]
print('Media (n per configurazione:', dict(df.groupby('config').size()), ')')
print(media.round(4).to_string())
print('\nErrore standard')
print(se.round(4).to_string())

In [ ]:
# Differenze appaiate rispetto al controllo k=0, sulle righe comuni
base = df[df.config == 'k0'].set_index('row_id')
diff = {}
for nome in ordine:
    if nome == 'k0':
        continue
    alt = df[df.config == nome].set_index('row_id')
    comuni = base.index.intersection(alt.index)
    d = alt.loc[comuni, METRICHE[:-1]] - base.loc[comuni, METRICHE[:-1]]
    diff[nome] = {m: (d[m].mean(), 1.96 * d[m].std() / np.sqrt(len(d))) for m in d}
    print(f'{nome:15s} n={len(comuni)}  ' + '  '.join(
        f'{m}: {mu:+.4f} ±{ic:.4f}' for m, (mu, ic) in diff[nome].items() if m in ('rouge1_f1', 'bertscore_f1', 'meteor')))

## Quasi-leakage dal train

Multi-News contiene notizie coperte più volte da cluster diversi: il vicino più simile del
train può essere la *stessa* notizia, con un riassunto umano quasi identico al riferimento.
Due domande: quanto spesso succede (similarità del 1° vicino) e quanto conta (ROUGE della
baseline retrieval-only in funzione della similarità).

In [ ]:
import matplotlib.pyplot as plt

sim1 = pd.Series({r['row_id']: VICINI[r['row_id']][0][1] for r in righe}, name='sim1')
ro = df[df.config == 'retrieval_only'].set_index('row_id')['rouge1_f1'].rename('rouge1_retrieval')
lk = pd.concat([sim1, ro], axis=1)
print(lk.describe().round(3).to_string())
for soglia in (0.8, 0.9):
    sotto = lk[lk.sim1 >= soglia]
    print(f'sim1 >= {soglia}: {len(sotto)} righe, ROUGE-1 F1 retrieval-only medio '
          f'{sotto.rouge1_retrieval.mean():.3f}')
print(f'Correlazione (Spearman) sim1 vs ROUGE retrieval-only: '
      f'{lk.sim1.corr(lk.rouge1_retrieval, method="spearman"):.3f}')

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(lk.sim1, lk.rouge1_retrieval, s=14, color='#d6336c')
ax.set_xlabel('similarita\' coseno del 1o vicino'); ax.set_ylabel('ROUGE-1 F1 retrieval-only')
ax.set_title('Il vicino del train come riassunto')
plt.tight_layout(); plt.show()

In [ ]:
# Riepilogo su disco (piccolo, committato con i TSV del pilota)
riepilogo = {'righe': 'val', 'n': N_PILOTA, 'seed': SEED, 'modello': MODELLO,
             'num_ctx': su.NUM_CTX_FEWSHOT, 'parole_esempio': su.PAROLE_ESEMPIO_FEWSHOT,
             'embedding': su.MODELLO_EMBEDDING_FEWSHOT,
             'media': media.round(5).to_dict(orient='index'),
             'errore_standard': se.round(5).to_dict(orient='index'),
             'diff_vs_k0': {c: {m: {'media': round(mu, 5), 'ic95': round(ic, 5)}
                                for m, (mu, ic) in v.items()} for c, v in diff.items()},
             'sim1': lk.sim1.describe().round(4).to_dict()}
with open(RISULTATI_PATH, 'w', encoding='utf-8') as f:
    json.dump(riepilogo, f, indent=2, ensure_ascii=False)
print(RISULTATI_PATH)

## Conclusione

*(da compilare dopo l'esecuzione: configurazione scelta per il notebook 21 e perché)*